In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

dq_rows = []  # collect (table, check, result, passed) tuples

def log_dq(table, check, result, passed):
    dq_rows.append((table, check, str(result), bool(passed)))

# ---------- PATIENTS ----------
patients_bronze = spark.table("workspace.bronze.patients")

patients_silver = (
    patients_bronze
    .withColumn("BIRTHDATE", F.to_date("BIRTHDATE"))
    .withColumn("DEATHDATE", F.to_date("DEATHDATE"))
    .withColumn("GENDER", F.upper(F.trim(F.col("GENDER"))))
    .withColumn("RACE", F.initcap(F.trim(F.col("RACE"))))
    .withColumn("ETHNICITY", F.initcap(F.trim(F.col("ETHNICITY"))))
    .dropDuplicates(["Id"])
    .filter(F.col("Id").isNotNull())
)

# DQ checks
total = patients_bronze.count()
nulls_id = patients_bronze.filter(F.col("Id").isNull()).count()
dupes = total - patients_bronze.select("Id").distinct().count()
log_dq("patients", "null_id_count", nulls_id, nulls_id == 0)
log_dq("patients", "duplicate_id_count", dupes, dupes == 0)
log_dq("patients", "future_birthdate_count",
       patients_silver.filter(F.col("BIRTHDATE") > F.current_date()).count(), True)

patients_silver.write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("workspace.silver.patients")

# ---------- ENCOUNTERS ----------
encounters_bronze = spark.table("workspace.bronze.encounters")

encounters_silver = (
    encounters_bronze
    .withColumn("START", F.to_timestamp("START"))
    .withColumn("STOP", F.to_timestamp("STOP"))
    .withColumn("ENCOUNTERCLASS", F.lower(F.trim(F.col("ENCOUNTERCLASS"))))
    .withColumn("LENGTH_OF_STAY_HOURS",
                (F.col("STOP").cast("long") - F.col("START").cast("long")) / 3600.0)
    .dropDuplicates(["Id"])
    .filter(F.col("PATIENT").isNotNull() & F.col("START").isNotNull())
)

nulls_patient = encounters_bronze.filter(F.col("PATIENT").isNull()).count()
bad_dates = encounters_silver.filter(F.col("STOP") < F.col("START")).count()
log_dq("encounters", "null_patient_fk_count", nulls_patient, nulls_patient == 0)
log_dq("encounters", "stop_before_start_count", bad_dates, bad_dates == 0)
log_dq("encounters", "valid_encounterclass_count",
       encounters_silver.filter(F.col("ENCOUNTERCLASS").isin(
           "inpatient","outpatient","ambulatory","wellness","urgentcare","emergency"
       )).count(), True)

encounters_silver.write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("workspace.silver.encounters")

# ---------- CONDITIONS ----------
conditions_silver = (
    spark.table("workspace.bronze.conditions")
    .withColumn("START", F.to_date("START"))
    .withColumn("STOP", F.to_date("STOP"))
    .filter(F.col("PATIENT").isNotNull() & F.col("ENCOUNTER").isNotNull())
    .dropDuplicates()
)
conditions_silver.write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("workspace.silver.conditions")

# ---------- MEDICATIONS ----------
medications_silver = (
    spark.table("workspace.bronze.medications")
    .withColumn("START", F.to_date("START"))
    .withColumn("STOP", F.to_date("STOP"))
    .filter(F.col("PATIENT").isNotNull())
    .dropDuplicates()
)
medications_silver.write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("workspace.silver.medications")

# ---------- PROCEDURES ----------
procedures_silver = (
    spark.table("workspace.bronze.procedures")
    .withColumn("START", F.to_timestamp("START"))
    .withColumn("STOP", F.to_timestamp("STOP"))
    .filter(F.col("PATIENT").isNotNull())
    .dropDuplicates()
)
procedures_silver.write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("workspace.silver.procedures")

# ---------- WRITE DQ RESULTS ----------
dq_schema = StructType([
    StructField("table_name", StringType()),
    StructField("check_name", StringType()),
    StructField("result", StringType()),
    StructField("passed", BooleanType()),
])
dq_df = spark.createDataFrame(dq_rows, schema=dq_schema) \
    .withColumn("run_timestamp", F.current_timestamp())

dq_df.write.mode("append").saveAsTable("workspace.silver.dq_results")

display(dq_df)

In [0]:
%sql
SHOW TABLES IN workspace.silver
